# Clase 14 — Práctica: Web Scraping con BeautifulSoup

Estos ejercicios refuerzan lo visto en la clase: navegar el árbol HTML con BeautifulSoup, filtrar con `find()`/`find_all()`, extraer una tabla a mano celda por celda, y extraer la misma tabla con `pd.read_html()`.

Todo el HTML de estos ejercicios está definido como string de Python — no necesitás conexión a internet.

Para cada ejercicio:
1. Lee el enunciado y las pistas.
2. Completa el código donde dice `# TODO`.
3. Ejecuta la celda y compará el resultado con lo esperado en el enunciado.
4. Si te trabas, abre la solución oculta debajo — pero intenta resolverlo por tu cuenta primero.

In [ ]:
from bs4 import BeautifulSoup
import pandas as pd
import io

## Ejercicio 1 — Navegando el perfil de un empleado (Tags, atributos, siblings)

Este es el HTML de una ficha de empleado ficticia:

```python
html_empleado = """
<div>
<h3 id="nombre_completo">Ana Torres</h3>
<p class="puesto">Puesto: Analista de Datos</p>
<p class="antiguedad">Antigüedad: 3 años</p>
</div>
"""
```

Completa el código para que:
1. Cree el objeto `BeautifulSoup` a partir de `html_empleado`, usando el parser `'html.parser'`.
2. Obtenga el tag `<h3>` y extraiga su **atributo `id`** con corchetes.
3. Obtenga el **texto** del `<h3>` (el nombre "Ana Torres") con `.string` o `.get_text()`.
4. A partir del `<h3>`, use `.find_next_sibling()` para llegar al `<p>` de "puesto", y desde ahí otra vez `.find_next_sibling()` para llegar al `<p>` de "antigüedad".

**Pistas:**
- El constructor es `BeautifulSoup(html, 'html.parser')`.
- Acceder a un tag por su nombre (`soup.h3`) te da el primero que aparece en el documento.
- El atributo se accede como un diccionario: `tag['id']`.
- Como el HTML original tiene saltos de línea entre los tags, `.next_sibling` "a secas" te puede devolver ese salto de línea en vez del tag que buscás. `.find_next_sibling()` salta directamente al **próximo tag**, ignorando espacios en blanco sueltos.

In [ ]:
html_empleado = """
<div>
<h3 id="nombre_completo">Ana Torres</h3>
<p class="puesto">Puesto: Analista de Datos</p>
<p class="antiguedad">Antigüedad: 3 años</p>
</div>
"""

# TODO: crea el objeto BeautifulSoup usando 'html.parser'
soup_empleado = None

# TODO: obtén el tag <h3> (acceso directo por nombre de tag)
tag_h3 = None
print("tag h3:", tag_h3)

# TODO: obtén el valor del atributo id del tag_h3, usando corchetes
id_h3 = None
print("id:", id_h3)

# TODO: obtén el texto contenido en tag_h3 (el nombre del empleado)
nombre_empleado = None
print("nombre:", nombre_empleado)

# TODO: usa find_next_sibling() sobre tag_h3 para llegar al <p> de "puesto"
tag_puesto = None
print("puesto:", tag_puesto)

# TODO: desde tag_puesto, usa find_next_sibling() para llegar al <p> de "antigüedad"
tag_antiguedad = None
print("antigüedad:", tag_antiguedad)

<details>
<summary>💡 Ver solución</summary>

```python
html_empleado = """
<div>
<h3 id="nombre_completo">Ana Torres</h3>
<p class="puesto">Puesto: Analista de Datos</p>
<p class="antiguedad">Antigüedad: 3 años</p>
</div>
"""

# Creamos el objeto BeautifulSoup con el parser incluido en la librería estándar
soup_empleado = BeautifulSoup(html_empleado, 'html.parser')

# soup.h3 devuelve el primer (y único) <h3> del documento
tag_h3 = soup_empleado.h3
print("tag h3:", tag_h3)

# Un tag se comporta como un diccionario para sus atributos
id_h3 = tag_h3['id']
print("id:", id_h3)

# El texto interno del tag es un NavigableString
nombre_empleado = tag_h3.get_text()
print("nombre:", nombre_empleado)

# find_next_sibling() avanza al PRÓXIMO TAG en el mismo nivel del árbol, saltando
# los saltos de línea/espacios en blanco que .next_sibling "a secas" sí devolvería
tag_puesto = tag_h3.find_next_sibling()
print("puesto:", tag_puesto)

tag_antiguedad = tag_puesto.find_next_sibling()
print("antigüedad:", tag_antiguedad)
```
</details>

## Ejercicio 2 — Filtrando reseñas de películas con `find_all()`

Este es el HTML de una lista de reseñas de películas:

```python
html_resenas = """
<div>
  <p class="resena" data-estrellas="5">Una obra maestra del cine.</p>
  <p class="resena" data-estrellas="2">No cumplió las expectativas.</p>
  <p class="resena" data-estrellas="5">La volvería a ver mil veces.</p>
  <p class="resena" data-estrellas="3">Entretenida pero olvidable.</p>
  <a href="https://ejemplo.com/pelicula/1">Más info</a>
  <a>Comentarios cerrados</a>
</div>
"""
```

Completa el código para que:
1. Cree el objeto `BeautifulSoup` a partir de `html_resenas`.
2. Use `find_all()` para obtener **todos** los `<p>` con `class_="resena"`.
3. Use `find_all()` con el atributo `data-estrellas="5"` para quedarte solo con las reseñas de 5 estrellas (pista: los atributos con guion se pasan como argumento con nombre usando `**{"data-estrellas": "5"}`, o directamente como diccionario en el parámetro `attrs`).
4. Use `find_all('a', href=True)` para encontrar los links que sí tienen `href`, y `find_all('a', href=False)` para los que no tienen.

**Pistas:**
- `class` es palabra reservada en Python, por eso se filtra con `class_` (guion bajo al final).
- Para atributos con guion en el nombre (como `data-estrellas`), no podés escribir `data-estrellas="5"` directamente como argumento de Python porque el guion no es válido en un nombre de parámetro; usá `attrs={"data-estrellas": "5"}`.
- `find_all('a', href=True)` filtra por "tiene el atributo", sin importar el valor exacto.

In [ ]:
html_resenas = """
<div>
  <p class="resena" data-estrellas="5">Una obra maestra del cine.</p>
  <p class="resena" data-estrellas="2">No cumplió las expectativas.</p>
  <p class="resena" data-estrellas="5">La volvería a ver mil veces.</p>
  <p class="resena" data-estrellas="3">Entretenida pero olvidable.</p>
  <a href="https://ejemplo.com/pelicula/1">Más info</a>
  <a>Comentarios cerrados</a>
</div>
"""

# TODO: crea el objeto BeautifulSoup a partir de html_resenas
soup_resenas = None

# TODO: encuentra todos los <p> con class_="resena"
todas_las_resenas = None
print("reseñas encontradas:", todas_las_resenas)

# TODO: encuentra solo las reseñas con atributo data-estrellas igual a "5"
# usa el parámetro attrs con un diccionario, ej: attrs={"data-estrellas": "5"}
resenas_5_estrellas = None
print("reseñas de 5 estrellas:", resenas_5_estrellas)

# TODO: encuentra los <a> que sí tienen href
links_con_href = None
print("links con href:", links_con_href)

# TODO: encuentra los <a> que NO tienen href
links_sin_href = None
print("links sin href:", links_sin_href)

<details>
<summary>💡 Ver solución</summary>

```python
html_resenas = """
<div>
  <p class="resena" data-estrellas="5">Una obra maestra del cine.</p>
  <p class="resena" data-estrellas="2">No cumplió las expectativas.</p>
  <p class="resena" data-estrellas="5">La volvería a ver mil veces.</p>
  <p class="resena" data-estrellas="3">Entretenida pero olvidable.</p>
  <a href="https://ejemplo.com/pelicula/1">Más info</a>
  <a>Comentarios cerrados</a>
</div>
"""

soup_resenas = BeautifulSoup(html_resenas, 'html.parser')

# class_ (con guion bajo) porque "class" es palabra reservada en Python
todas_las_resenas = soup_resenas.find_all('p', class_="resena")
print("cantidad de reseñas:", len(todas_las_resenas))

# Los atributos con guion en el nombre se pasan dentro de un diccionario en attrs
resenas_5_estrellas = soup_resenas.find_all('p', attrs={"data-estrellas": "5"})
print("reseñas de 5 estrellas:")
for r in resenas_5_estrellas:
    print("-", r.get_text())

# href=True filtra "tiene el atributo href", sin importar el valor
links_con_href = soup_resenas.find_all('a', href=True)
print("links con href:", links_con_href)

# href=False filtra los que NO tienen el atributo
links_sin_href = soup_resenas.find_all('a', href=False)
print("links sin href:", links_sin_href)
```
</details>

## Ejercicio 3 — Una tabla de empleados: a mano vs. `pd.read_html()`

Este es el HTML de una tabla de empleados de una empresa ficticia:

```python
html_tabla_empleados = """
<table id="tabla_staff">
  <tr>
    <th>Nombre</th>
    <th>Departamento</th>
    <th>Salario</th>
  </tr>
  <tr>
    <td>Ana Torres</td>
    <td>Datos</td>
    <td>52000</td>
  </tr>
  <tr>
    <td>Luis Peña</td>
    <td>Ventas</td>
    <td>41000</td>
  </tr>
  <tr>
    <td>Marta Ruiz</td>
    <td>Datos</td>
    <td>58000</td>
  </tr>
</table>
"""
```

Completa el código para que:
1. **A mano con BeautifulSoup**: cree el `soup`, encuentre la tabla con `find('table')`, recorra sus filas con `find_all('tr')` (saltando el encabezado) y, para cada fila, extraiga nombre, departamento y salario en una lista de tuplas llamada `empleados`.
2. Convierta esa lista de tuplas `empleados` en un DataFrame de pandas con columnas `["Nombre", "Departamento", "Salario"]`, llamado `df_manual`.
3. **Con pandas directo**: use `pd.read_html(io.StringIO(html_tabla_empleados))` para obtener la misma tabla en un DataFrame llamado `df_pandas`, sin recorrer nada a mano.
4. Compare ambos DataFrames imprimiendo sus columnas y su cantidad de filas — deberían coincidir en contenido, aunque `df_pandas` va a traer los encabezados ya usados como nombres de columna automáticamente.

**Pistas:**
- El patrón de siempre: `tabla.find_all('tr')` para las filas, y dentro de cada fila `fila.find_all('td')` para las celdas.
- La primera fila es el encabezado ("Nombre", "Departamento", "Salario"), no un empleado — hay que saltarla al recorrer (por ejemplo con `filas[1:]`).
- `pd.DataFrame(lista_de_tuplas, columns=[...])` arma un DataFrame a partir de una lista de tuplas.
- `pd.read_html()` moderno recomienda envolver el string HTML con `io.StringIO(...)`.

In [ ]:
html_tabla_empleados = """
<table id="tabla_staff">
  <tr>
    <th>Nombre</th>
    <th>Departamento</th>
    <th>Salario</th>
  </tr>
  <tr>
    <td>Ana Torres</td>
    <td>Datos</td>
    <td>52000</td>
  </tr>
  <tr>
    <td>Luis Peña</td>
    <td>Ventas</td>
    <td>41000</td>
  </tr>
  <tr>
    <td>Marta Ruiz</td>
    <td>Datos</td>
    <td>58000</td>
  </tr>
</table>
"""

# TODO: crea el soup con 'html.parser'
soup_staff = None

# TODO: encuentra la tabla con find('table')
tabla_staff = None

# TODO: obtén todas las filas con find_all('tr')
filas_staff = None

# TODO: recorre las filas SALTANDO el encabezado (filas_staff[1:]) y arma
# una lista de tuplas (nombre, departamento, salario) llamada 'empleados'
empleados = []
# tu bucle for acá

print("empleados extraídos a mano:", empleados)

# TODO: convierte 'empleados' en un DataFrame llamado df_manual
# con columns=["Nombre", "Departamento", "Salario"]
df_manual = None
print(df_manual)

In [ ]:
# TODO: usa pd.read_html(io.StringIO(html_tabla_empleados)) para obtener
# la lista de tablas, y guarda la primera en df_pandas
tablas_staff = None
df_pandas = None
print(df_pandas)

# TODO: imprime las columnas y la cantidad de filas de ambos DataFrames para compararlos
print("columnas df_manual:", None)
print("columnas df_pandas:", None)
print("filas df_manual:", None)
print("filas df_pandas:", None)

<details>
<summary>💡 Ver solución</summary>

```python
html_tabla_empleados = """
<table id="tabla_staff">
  <tr>
    <th>Nombre</th>
    <th>Departamento</th>
    <th>Salario</th>
  </tr>
  <tr>
    <td>Ana Torres</td>
    <td>Datos</td>
    <td>52000</td>
  </tr>
  <tr>
    <td>Luis Peña</td>
    <td>Ventas</td>
    <td>41000</td>
  </tr>
  <tr>
    <td>Marta Ruiz</td>
    <td>Datos</td>
    <td>58000</td>
  </tr>
</table>
"""

soup_staff = BeautifulSoup(html_tabla_empleados, 'html.parser')

# find() nos da la única tabla del documento
tabla_staff = soup_staff.find('table')

# find_all('tr') trae todas las filas, incluyendo el encabezado
filas_staff = tabla_staff.find_all('tr')

empleados = []
for fila in filas_staff[1:]:   # [1:] salta la fila de encabezado
    cols = fila.find_all('td')
    nombre = cols[0].get_text()
    departamento = cols[1].get_text()
    salario = cols[2].get_text()
    empleados.append((nombre, departamento, salario))

print("empleados extraídos a mano:", empleados)

df_manual = pd.DataFrame(empleados, columns=["Nombre", "Departamento", "Salario"])
print(df_manual)
```

```python
import io

# pd.read_html hace todo el trabajo de detectar y armar el DataFrame por nosotros
tablas_staff = pd.read_html(io.StringIO(html_tabla_empleados))
df_pandas = tablas_staff[0]
print(df_pandas)

# Comparamos ambos resultados
print("columnas df_manual:", list(df_manual.columns))
print("columnas df_pandas:", list(df_pandas.columns))
print("filas df_manual:", len(df_manual))
print("filas df_pandas:", len(df_pandas))
# Ambos DataFrames traen los mismos 3 empleados; pd.read_html además detectó
# automáticamente que la primera fila de la tabla era el encabezado, y la usó
# como nombres de columna sin que tuviéramos que indicárselo.
```
</details>